# Hybrid Local AI + Gemini Medicine Identification System: Model Training & Evaluation

## Project Architecture Overview

This notebook trains, evaluates, and exports the **Local Machine Learning Medicine Matching Pipeline** for an AI-powered Indian medicine identification platform. The system operates as a **hybrid edge-cloud architecture** designed to minimize latency, reduce cloud API token costs, and provide reliable, grounded pharmaceutical guidance.

### System Pipeline
```
USER UPLOADS MEDICINE PHOTO
          ↓
IMAGE VALIDATION & PREPROCESSING (OpenCV / Pillow)
          ↓
LOCAL OCR TEXT EXTRACTION (PyTesseract / Tesseract 5.5 on Apple Silicon)
          ↓
TEXT NORMALIZATION & ATTRIBUTE PARSING (Strength, Dosage Form, Active Salts)
          ↓
LOCAL ML MATCHING MODEL (Character & Word TF-IDF n-grams + Cosine Retrieval)
          ↓
MULTI-ATTRIBUTE VERIFICATION & ABSTENTION POLICY
          ├── [Confidence ≥ 0.82 & Unambiguous & Verified] ──> Return Local Verified Match (No Gemini Call)
          ├── [Ambiguous / Low Confidence / Conflict] ──────> Fallback to Gemini with OCR Hints
          └── [Unreadable / Unmarked Pill / Blurry] ─────────> Return Safe Rejection with Guidance
```

### Key Functional Distinctions
- **OCR (Optical Character Recognition)**: Converts raw raster packaging pixels into candidate character sequences with word-level confidence. Does *not* determine clinical identity.
- **Medicine Catalogue Lookup**: Fast in-memory retrieval of 20,485+ verified Indian pharmaceutical formulations with CDSCO provenance, active salts, and therapeutic classifications.
- **Machine Learning Matcher**: A trainable character (2–5) and word n-gram TF-IDF vectorizer that tolerates OCR optical confusion (e.g., `0` vs `O`, `1` vs `I`, punctuation drops) and ranks candidate formulations.
- **Abstention Policy**: A calibrated decision gate that checks strength consistency, dosage form agreement, and ambiguity margins before accepting a candidate.
- **Generative AI (Gemini 2.5 / 2.0 Flash)**: Used strictly as a fallback when packaging is degraded, partially torn, or ambiguous.

### Model Limitations
- Cannot establish chemical authenticity of counterfeit physical pills.
- Requires readable text on blister packs, cartons, or bottles.
- Abstains on foreign medicines not licensed in the Indian CDSCO catalogue.


In [1]:
# Notebook Cell 2: Environment Setup & Hardware Inspection
import sys
import os
import platform
import json
import time
import re
import csv
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import PIL
from PIL import Image, ImageOps, ImageDraw, ImageFont
import cv2
import pytesseract

# Configure Tesseract binary path for macOS Apple Silicon / Homebrew
tesseract_bin = "/opt/homebrew/bin/tesseract"
if os.path.exists(tesseract_bin):
    pytesseract.pytesseract.tesseract_cmd = tesseract_bin

print("=== Environment Information ===")
print(f"Python Version: {sys.version.split()[0]}")
print(f"Platform: {platform.system()} ({platform.machine()}) - Apple Silicon Optimized")
print(f"PyTesseract Version: {pytesseract.get_tesseract_version()}")
print(f"Scikit-Learn Version: {sklearn.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version: {np.__version__}")
print(f"OpenCV Version: {cv2.__version__}")
print("Hardware acceleration: CPU-first baseline active (no CUDA required).")


=== Environment Information ===
Python Version: 3.11.15
Platform: Darwin (arm64) - Apple Silicon Optimized
PyTesseract Version: 5.5.3
Scikit-Learn Version: 1.9.1
Pandas Version: 3.0.6
NumPy Version: 2.4.6
OpenCV Version: 5.0.0
Hardware acceleration: CPU-first baseline active (no CUDA required).


In [2]:
# Notebook Cell 3: Dataset Loading & Validation
catalog_path = os.path.abspath(os.path.join("..", "..", "data", "medicines", "medicine_catalog.csv"))
if not os.path.exists(catalog_path):
    catalog_path = os.path.abspath(os.path.join("data", "medicines", "medicine_catalog.csv"))

df_catalog = pd.read_csv(catalog_path)
print(f"Loaded verified medicine catalogue from: {catalog_path}")
print(f"Total Catalogue Records: {len(df_catalog):,}")

# Validate required schema columns
required_columns = [
    'product_id', 'brand_name', 'generic_name', 'active_ingredients',
    'strength', 'dosage_form', 'manufacturer', 'aliases',
    'common_uses', 'precautions', 'prescription_status',
    'source_name', 'source_url', 'verified_at'
]
missing_cols = [c for c in required_columns if c not in df_catalog.columns]
assert not missing_cols, f"Missing required columns: {missing_cols}"
print("Schema Validation: All 14 required columns present.")

# Check nulls and uniqueness
print("\nMissing Value Analysis:")
print(df_catalog[['brand_name', 'dosage_form', 'prescription_status']].isnull().sum())

print("\nTop 5 Dosage Forms:")
print(df_catalog['dosage_form'].value_counts().head(5))

print("\nTop 5 Prescription Status Distribution:")
print(df_catalog['prescription_status'].value_counts())

# Document optional packaging image dataset schema
image_dir = os.path.abspath(os.path.join("..", "..", "data", "medicines", "images"))
print(f"\nPackaging image repository path: {image_dir}")
print("Image metadata schema: [image_id, filename, product_id, view_type (front/back/foil), lighting_condition, is_readable]")


Loaded verified medicine catalogue from: /Users/kirpalsingh/Desktop/Khushi medicine project/data/medicines/medicine_catalog.csv
Total Catalogue Records: 20,485
Schema Validation: All 14 required columns present.

Missing Value Analysis:
brand_name             0
dosage_form            0
prescription_status    0
dtype: int64

Top 5 Dosage Forms:
dosage_form
Tablet       11657
Injection     1837
Syrup         1729
Capsule       1640
Cream          464
Name: count, dtype: int64

Top 5 Prescription Status Distribution:
prescription_status
Prescription    16522
OTC              3963
Name: count, dtype: int64

Packaging image repository path: /Users/kirpalsingh/data/medicines/images
Image metadata schema: [image_id, filename, product_id, view_type (front/back/foil), lighting_condition, is_readable]


In [3]:
# Notebook Cell 4: Data Cleaning and Text Normalization
def clean_text(text: str) -> str:
    """Standardize whitespace and punctuation while preserving strength units."""
    if not text or pd.isna(text):
        return ""
    s = str(text).lower()
    s = re.sub(r"[^a-z0-9.%\/]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def extract_strengths(text: str) -> list[str]:
    """Extract strength expressions like 650 mg, 120mg, 20% w/w, 1 mg / 500 mg SR."""
    matches = re.findall(
        r"\b\d+(?:\.\d+)?(?:\s*\/\s*\d+(?:\.\d+)?)*(?:\s*(?:mg|mcg|gm|g|ml|iu|%|\s*w\/w|\s*w\/v))\b",
        text,
        flags=re.IGNORECASE
    )
    return [m.lower().replace(" ", "") for m in matches]

def normalize_dosage_form(raw_form: str) -> str:
    """Map abbreviated or misrecognized dosage forms to standard forms."""
    form_map = {
        'tab': 'Tablet', 'tabs': 'Tablet', 'tablets': 'Tablet',
        'cap': 'Capsule', 'caps': 'Capsule', 'capsules': 'Capsule',
        'syp': 'Syrup', 'susp': 'Suspension', 'inj': 'Injection',
        'gel': 'Gel', 'crm': 'Cream', 'oint': 'Ointment', 'oinment': 'Ointment',
        'rotacaps': 'Rotacap', 'respules': 'Respule'
    }
    clean = clean_text(raw_form)
    return form_map.get(clean, raw_form.capitalize() if raw_form else 'Tablet')

def handle_ocr_confusions(text: str) -> list[str]:
    """Generate potential candidate hints for common optical character confusions (e.g. O vs 0, I vs 1)."""
    hints = [text]
    # In strength numbers, O often misread for 0
    sub_0 = re.sub(r"(\d+)o\b", r"\g<1>0", text, flags=re.IGNORECASE)
    if sub_0 != text:
        hints.append(sub_0)
    sub_l = re.sub(r"\b([a-z]+)1\b", r"\g<1>l", text, flags=re.IGNORECASE)
    if sub_l != text:
        hints.append(sub_l)
    return list(set(hints))

# Demonstration of normalization
sample_raw = "Paracip 650mg TAB. IP (Paracetamol 650 mg) 10's Blister"
print("Raw Packaging String:", sample_raw)
print("Normalized Text:", clean_text(sample_raw))
print("Extracted Strengths:", extract_strengths(sample_raw))
print("Extracted Dosage Form:", normalize_dosage_form("TAB"))
print("OCR Confusion Hints:", handle_ocr_confusions("Paracip 65Omg"))


Raw Packaging String: Paracip 650mg TAB. IP (Paracetamol 650 mg) 10's Blister
Normalized Text: paracip 650mg tab. ip paracetamol 650 mg 10 s blister
Extracted Strengths: ['650mg', '650mg']
Extracted Dosage Form: Tablet
OCR Confusion Hints: ['Paracip 65Omg']


In [4]:
# Notebook Cell 5: Local OCR Pipeline & Preprocessing Wrapper
class LocalOCRPipeline:
    def __init__(self):
        self.tesseract_cmd = pytesseract.pytesseract.tesseract_cmd

    def preprocess_image(self, image: Image.Image, max_dim: int = 1800) -> np.ndarray:
        # Correct orientation
        try:
            image = ImageOps.exif_transpose(image)
        except Exception:
            pass

        # Resize if oversized
        w, h = image.size
        if max(w, h) > max_dim:
            scale = max_dim / float(max(w, h))
            image = image.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)

        # Grayscale and CLAHE contrast enhancement
        rgb = np.array(image.convert("RGB"))
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        return clahe.apply(gray)

    def extract_text_and_confidence(self, image: Image.Image) -> dict:
        processed = self.preprocess_image(image)
        custom_config = r"--oem 3 --psm 11"
        data = pytesseract.image_to_data(processed, output_type=pytesseract.Output.DICT, config=custom_config)
        
        words = []
        confs = []
        for i in range(len(data["text"])):
            w = data["text"][i].strip()
            conf = float(data["conf"][i])
            if w and conf >= 0:
                words.append(w)
                confs.append(conf)

        avg_conf = (sum(confs) / len(confs) / 100.0) if confs else 0.0
        full_text = " ".join(words).strip()
        is_readable = bool(full_text) and len(full_text) >= 4 and avg_conf >= 0.25

        return {
            "text": full_text,
            "confidence": round(avg_conf, 3),
            "word_count": len(words),
            "is_readable": is_readable
        }

ocr_pipe = LocalOCRPipeline()

# Test 1: Synthesize a clear medicine blister packaging image in memory
synth_img = Image.new("RGB", (600, 200), color=(250, 250, 250))
draw = ImageDraw.Draw(synth_img)
draw.rectangle([(10, 10), (590, 190)], outline=(180, 180, 180), width=2)
draw.text((30, 30), "Allegra 120mg Tablet", fill=(10, 10, 10))
draw.text((30, 70), "Fexofenadine HCl Tablets IP 120 mg", fill=(40, 40, 40))
draw.text((30, 110), "Sanofi India Limited | 10 Tablets", fill=(70, 70, 70))
draw.text((30, 150), "SCHEDULE H PRESCRIPTION DRUG", fill=(160, 20, 20))

res_clear = ocr_pipe.extract_text_and_confidence(synth_img)
print("Clear Packaging OCR Test:")
print(f"Extracted Text: '{res_clear['text']}'")
print(f"Word Confidence: {res_clear['confidence']} (Readable: {res_clear['is_readable']})")

# Test 2: Blank / Unreadable image test
blank_img = Image.new("RGB", (300, 300), color=(128, 128, 128))
res_blank = ocr_pipe.extract_text_and_confidence(blank_img)
print("\nUnreadable Blank Image Test:")
print(f"Extracted Text: '{res_blank['text']}'")
print(f"Confidence: {res_blank['confidence']} (Readable: {res_blank['is_readable']})")


Clear Packaging OCR Test:
Extracted Text: '‘Allegra 120mg Tablet Fexofenadine HCl Tablets P 120 mg Sanofi India Limited | 10 Tablets SCHEDULE H PRESCRIPTION DRUG'
Word Confidence: 0.693 (Readable: True)

Unreadable Blank Image Test:
Extracted Text: ''
Confidence: 0.0 (Readable: False)


In [5]:
# Notebook Cell 6: Baseline Medicine Matching Model
records = df_catalog.to_dict('records')

# Build corpus representations combining brand names, aliases, and active salts
corpus = []
for r in records:
    brand = str(r.get("brand_name", ""))
    generic = str(r.get("generic_name", ""))
    salts = str(r.get("active_ingredients", ""))
    aliases = str(r.get("aliases", ""))
    doc = f"{brand} {brand} {aliases} {generic} {salts}".strip()
    corpus.append(clean_text(doc))

print(f"Constructing character n-gram TF-IDF index over {len(corpus):,} catalogue documents...")
vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    max_features=60000,
    sublinear_tf=True
)

catalog_tfidf = vectorizer.fit_transform(corpus)
print(f"TF-IDF Matrix Shape: {catalog_tfidf.shape} (Documents: {catalog_tfidf.shape[0]:,}, Features: {catalog_tfidf.shape[1]:,})")

def rank_candidates(query: str, top_k: int = 5) -> list[dict]:
    clean_q = clean_text(query)
    if not clean_q or len(clean_q) < 2:
        return []
    
    q_vec = vectorizer.transform([clean_q])
    sims = cosine_similarity(q_vec, catalog_tfidf).flatten()
    
    initial_k = min(top_k * 4, len(sims))
    top_indices = np.argpartition(sims, -initial_k)[-initial_k:]
    top_indices = top_indices[np.argsort(-sims[top_indices])]
    
    query_strengths = extract_strengths(query)
    
    candidates = []
    for idx in top_indices:
        sim = float(sims[idx])
        if sim < 0.05:
            continue
        rec = records[idx]
        cand_str = str(rec.get("strength", ""))
        
        # Attribute verification adjustment
        adj_score = sim
        str_status = "neutral"
        if cand_str and query_strengths:
            cand_clean = cand_str.lower().replace(" ", "")
            cand_nums = set(re.findall(r"\d+(?:\.\d+)?", cand_clean))
            q_nums = set(re.findall(r"\d+(?:\.\d+)?", " ".join(query_strengths)))
            if cand_nums and cand_nums.issubset(q_nums):
                adj_score += 0.12
                str_status = "matched"
            elif cand_nums and cand_nums.isdisjoint(q_nums):
                adj_score -= 0.35
                str_status = "mismatched"

        # Brand name match bonus
        brand_clean = clean_text(rec.get("brand_name", ""))
        if brand_clean in clean_q or clean_q in brand_clean:
            adj_score += 0.15

        calib_score = round(max(0.0, min(0.99, adj_score)), 3)
        candidates.append({
            "product_id": rec.get("product_id"),
            "brand_name": rec.get("brand_name"),
            "generic_name": rec.get("generic_name"),
            "strength": rec.get("strength"),
            "dosage_form": rec.get("dosage_form"),
            "calibrated_score": calib_score,
            "strength_status": str_status
        })

    candidates.sort(key=lambda x: x["calibrated_score"], reverse=True)
    return candidates[:top_k]

# Test ranking
sample_cands = rank_candidates("Allegra 120mg Tablet Sanofi 10'S", top_k=3)
for c in sample_cands:
    print(f"Candidate: {c['brand_name']} | Strength: {c['strength']} | Score: {c['calibrated_score']}")


Constructing character n-gram TF-IDF index over 20,485 catalogue documents...


TF-IDF Matrix Shape: (20485, 60000) (Documents: 20,485, Features: 60,000)
Candidate: Allegra 120mg Tablet | Strength: 120 mg | Score: 0.846
Candidate: Allegra 120mg Tablet 10'S | Strength: 120mg | Score: 0.699
Candidate: Galegra 120mg Tablet 6'S | Strength: 120mg | Score: 0.55


# Notebook Cell 7: Optional Visual Matching Architecture & Vision Embeddings

### Visual Matching vs Text-Guided Catalogue Retrieval
While text-guided OCR + TF-IDF candidate matching achieves high precision on printed pharmaceutical packaging, physical packaging images may also be retrieved using dense image-text embeddings (such as OpenAI CLIP, BioCLIP, or a lightweight MobileNet feature extractor).

**Key Architectural Role**:
- A vision model proposes candidate brand families based on blister foil layout, logo contours, and box typography.
- **Safety Boundary**: Visual embeddings alone **must never** establish drug dosage, active ingredients, or clinical safety because two medicines with identical foil blisters (e.g. Paracetamol 500mg vs Paracetamol 650mg) have nearly identical visual embeddings but distinct therapeutic dosages.
- Visual matching is maintained as an optional candidate retrieval layer, while the verified catalogue index performs deterministic attribute validation.


In [6]:
# Notebook Cell 7 (Code): Visual Matching Interface Specification
class OptionalVisualEmbeddingModel:
    """
    Interface for visual candidate retrieval using dense image embeddings.
    Kept modular so that when a physical packaging dataset is ingested,
    feature vectors can be indexed in FAISS or Scikit-Learn NearestNeighbors.
    """
    def __init__(self, embedding_dim: int = 512):
        self.embedding_dim = embedding_dim
        self.is_enabled = False

    def extract_image_features(self, image: Image.Image) -> np.ndarray:
        """Extract L2-normalized visual feature vector from medicine packaging image."""
        # Placeholder for pretrained vision encoder (e.g. CLIP / MobileNet)
        np.random.seed(42)
        feat = np.random.randn(self.embedding_dim).astype(np.float32)
        return feat / np.linalg.norm(feat)

    def retrieve_visual_candidates(self, image_features: np.ndarray, top_k: int = 5) -> list:
        return []

visual_model = OptionalVisualEmbeddingModel()
print(f"Visual Retrieval Engine Initialized (Status: Optional / Standby, Dim: {visual_model.embedding_dim})")


Visual Retrieval Engine Initialized (Status: Optional / Standby, Dim: 512)


In [7]:
# Notebook Cell 8: Model Evaluation & Benchmark Testing
# Create a diverse evaluation test suite of real Indian medicines with simulated packaging OCR noise
eval_test_cases = [
    {
        "id": "TC-01",
        "name": "Allegra 120mg Tablet",
        "ocr_input": "Allegra 120mg Tablet 10'S Fexofenadine Hydrochloride 120 mg Sanofi",
        "expected_brand": "Allegra",
        "expected_strength": "120",
        "should_abstain": False
    },
    {
        "id": "TC-02",
        "name": "Aziderm 20% Cream",
        "ocr_input": "Aziderm 20% Cream 15gm Azelaic Acid Micro Labs",
        "expected_brand": "Aziderm",
        "expected_strength": "20%",
        "should_abstain": False
    },
    {
        "id": "TC-03",
        "name": "Glycomet GP 1 Tablet",
        "ocr_input": "Glycomet GP 1 Tablet Glimepiride 1 mg Metformin 500 mg SR USV",
        "expected_brand": "Glycomet",
        "expected_strength": "500",
        "should_abstain": False
    },
    {
        "id": "TC-04",
        "name": "Foracort 200 Rotacap",
        "ocr_input": "Foracort 200 Rotacap Formoterol 6 mcg Budesonide 200 mcg Cipla",
        "expected_brand": "Foracort",
        "expected_strength": "200",
        "should_abstain": False
    },
    {
        "id": "TC-05",
        "name": "Paracip 650 Tablet",
        "ocr_input": "Paracip 650 Tablet Paracetamol IP 650 mg Cipla",
        "expected_brand": "Paracip",
        "expected_strength": "650",
        "should_abstain": False
    },
    {
        "id": "TC-06",
        "name": "Allegra 180mg Tablet (Strength Distinction)",
        "ocr_input": "Allegra 180mg Tablet 10'S Fexofenadine 180 mg Sanofi",
        "expected_brand": "Allegra",
        "expected_strength": "180",
        "should_abstain": False
    },
    {
        "id": "TC-07",
        "name": "Optical Confusion: 0 vs O (Paracip 65Omg)",
        "ocr_input": "Paracip 65Omg Tablet Paracetamol IP Cipla",
        "expected_brand": "Paracip",
        "expected_strength": "650",
        "should_abstain": False
    },
    {
        "id": "TC-08",
        "name": "Unknown Brand (Safe Abstention)",
        "ocr_input": "Xylopharma 999 Special Drug Formulation Unknown Laboratories",
        "expected_brand": None,
        "expected_strength": None,
        "should_abstain": True
    },
    {
        "id": "TC-09",
        "name": "Unreadable Glare / Unmarked Foil (Rejection)",
        "ocr_input": "[UNREADABLE IMAGE EVIDENCE: Loose round white unmarked pill with heavy camera glare]",
        "expected_brand": None,
        "expected_strength": None,
        "should_abstain": True
    }
]

eval_results = []
correct_top1 = 0
correct_top3 = 0
correct_abstentions = 0
total_cases = len(eval_test_cases)

for tc in eval_test_cases:
    cands = rank_candidates(tc["ocr_input"], top_k=3)
    top = cands[0] if cands else None
    
    top1_match = False
    top3_match = False
    
    if tc["should_abstain"]:
        # Safe behavior: score should be low (< 0.70) or no match
        abstained = (not top) or (top["calibrated_score"] < 0.75)
        if abstained:
            correct_abstentions += 1
            top1_match = True
    else:
        if top and tc["expected_brand"].lower() in top["brand_name"].lower():
            top1_match = True
            correct_top1 += 1
        top3_match = any(tc["expected_brand"].lower() in c["brand_name"].lower() for c in cands)
        if top3_match:
            correct_top3 += 1

    eval_results.append({
        "case_id": tc["id"],
        "name": tc["name"],
        "top_candidate": top["brand_name"] if top else "None",
        "score": top["calibrated_score"] if top else 0.0,
        "passed": top1_match
    })

df_eval = pd.DataFrame(eval_results)
print("=== Benchmark Evaluation Summary ===")
print(df_eval.to_string(index=False))

identifiable_cases = sum(1 for tc in eval_test_cases if not tc["should_abstain"])
top1_acc = (correct_top1 / identifiable_cases) * 100
top3_rec = (correct_top3 / identifiable_cases) * 100
abstention_acc = (correct_abstentions / (total_cases - identifiable_cases)) * 100

print(f"\nTop-1 Identification Accuracy on Clear Packaging: {top1_acc:.1f}%")
print(f"Top-3 Candidate Recall: {top3_rec:.1f}%")
print(f"Abstention / Safety Rejection Accuracy: {abstention_acc:.1f}%")

# Save report
report_dir = os.path.abspath(os.path.join("..", "..", "ml", "reports"))
if not os.path.exists(report_dir):
    report_dir = os.path.abspath(os.path.join("ml", "reports"))
os.makedirs(report_dir, exist_ok=True)

report_data = {
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "total_test_cases": total_cases,
    "top1_accuracy_percent": top1_acc,
    "top3_recall_percent": top3_rec,
    "abstention_accuracy_percent": abstention_acc,
    "cases": eval_results
}

report_file = os.path.join(report_dir, "evaluation_report.json")
with open(report_file, "w") as f:
    json.dump(report_data, f, indent=2)
print(f"Saved evaluation report to: {report_file}")


=== Benchmark Evaluation Summary ===
case_id                                         name             top_candidate  score  passed
  TC-01                         Allegra 120mg Tablet      Allegra 120mg Tablet  0.990    True
  TC-02                            Aziderm 20% Cream    Aziderm 20% Cream 15gm  0.990    True
  TC-03                         Glycomet GP 1 Tablet             Glycomet GP 1  0.990    True
  TC-04                         Foracort 200 Rotacap      Foracort 200 Rotacap  0.990    True
  TC-05                           Paracip 650 Tablet               Paracip 650  0.990    True
  TC-06  Allegra 180mg Tablet (Strength Distinction) Allegra 180mg Tablet 10'S  0.990    True
  TC-07    Optical Confusion: 0 vs O (Paracip 65Omg)               Paracip 650  0.782    True
  TC-08              Unknown Brand (Safe Abstention)                       999  0.373    True
  TC-09 Unreadable Glare / Unmarked Foil (Rejection)     Glare 2mg Tablet 10'S  0.263    True

Top-1 Identification A

In [8]:
# Notebook Cell 9: Confidence Calibration and Abstention Policy
def evaluate_decision(candidates: list[dict], query_text: str = "") -> dict:
    """
    Decision gate: Determines whether a candidate can be accepted locally or must trigger Gemini.
    """
    if not candidates:
        return {
            "decision": "UNIDENTIFIED",
            "should_fallback_to_gemini": True,
            "top_candidate": None,
            "confidence_score": 0.10,
            "reason": "No medicine candidates matched the extracted packaging text."
        }

    top = candidates[0]
    score = top["calibrated_score"]
    norm_q = clean_text(query_text)

    # Strength conflict check
    if top.get("strength_status") == "mismatched":
        return {
            "decision": "ABSTAIN_STRENGTH_CONFLICT",
            "should_fallback_to_gemini": True,
            "top_candidate": top,
            "confidence_score": round(max(0.2, score * 0.5), 2),
            "reason": f"Strength mismatch detected between packaging and candidate ({top['brand_name']})."
        }

    # Ambiguity check: compare against candidate from a conflicting formulation or competing brand
    conflicting_candidate = None
    for second in candidates[1:]:
        norm_top = clean_text(top.get("brand_name", ""))
        norm_sec = clean_text(second.get("brand_name", ""))
        
        # Clean strength numbers
        str_top_val = "" if pd.isna(top.get("strength")) else str(top.get("strength") or "")
        str_sec_val = "" if pd.isna(second.get("strength")) else str(second.get("strength") or "")
        nums_top = set(re.findall(r"\d+(?:\.\d+)?", str_top_val))
        nums_sec = set(re.findall(r"\d+(?:\.\d+)?", str_sec_val))
        same_strength = (nums_top == nums_sec) or (nums_top and nums_sec and not nums_top.isdisjoint(nums_sec))
        
        brand_top = norm_top.split()[0] if norm_top else ""
        brand_sec = norm_sec.split()[0] if norm_sec else ""
        same_brand_family = (brand_top == brand_sec and len(brand_top) >= 3)

        if same_brand_family and same_strength:
            continue

        if norm_q and brand_top and brand_sec:
            if brand_top in norm_q and brand_sec not in norm_q:
                continue

        conflicting_candidate = second
        break

    if conflicting_candidate:
        margin = score - conflicting_candidate["calibrated_score"]
        if margin < 0.08 and score >= 0.65:
            return {
                "decision": "ABSTAIN_AMBIGUOUS",
                "should_fallback_to_gemini": True,
                "top_candidate": top,
                "confidence_score": round(score, 2),
                "reason": f"Ambiguous formulation match between '{top['brand_name']}' and '{conflicting_candidate['brand_name']}'."
            }

    if score >= 0.82:
        return {
            "decision": "LOCAL_IDENTIFICATION_ACCEPTED",
            "should_fallback_to_gemini": False,
            "top_candidate": top,
            "confidence_score": round(score, 2),
            "reason": "High-confidence local match verified against Indian pharmaceutical catalog."
        }

    return {
        "decision": "ABSTAIN_LOW_CONFIDENCE",
        "should_fallback_to_gemini": True,
        "top_candidate": top,
        "confidence_score": round(score, 2),
        "reason": f"Local confidence ({score:.2f}) below acceptance threshold (0.82)."
    }

print("Abstention Policy Demo:")
for q in ["Allegra 120mg Tablet Sanofi", "Paracip 500mg Tablet", "Unknown Medicine"]:
    c = rank_candidates(q, top_k=3)
    d = evaluate_decision(c, query_text=q)
    print(f"Query: '{q}' -> Decision: {d['decision']} (Confidence: {d['confidence_score']}) -> Fallback to Gemini: {d['should_fallback_to_gemini']}")


Abstention Policy Demo:
Query: 'Allegra 120mg Tablet Sanofi' -> Decision: LOCAL_IDENTIFICATION_ACCEPTED (Confidence: 0.85) -> Fallback to Gemini: False
Query: 'Paracip 500mg Tablet' -> Decision: ABSTAIN_LOW_CONFIDENCE (Confidence: 0.74) -> Fallback to Gemini: True
Query: 'Unknown Medicine' -> Decision: ABSTAIN_LOW_CONFIDENCE (Confidence: 0.28) -> Fallback to Gemini: True


In [9]:
# Notebook Cell 10: Save and Reload Model Artifacts
model_output_dir = os.path.abspath(os.path.join("..", "..", "ml", "models", "medicine_matcher"))
if not os.path.exists(model_output_dir):
    model_output_dir = os.path.abspath(os.path.join("ml", "models", "medicine_matcher"))
os.makedirs(model_output_dir, exist_ok=True)

# 1. Save Vectorizer
joblib.dump(vectorizer, os.path.join(model_output_dir, "vectorizer.joblib"))

# 2. Save TF-IDF matrix
joblib.dump(catalog_tfidf, os.path.join(model_output_dir, "catalog_tfidf.joblib"))

# 3. Save Catalog Records
with open(os.path.join(model_output_dir, "catalog_records.json"), "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False)

# 4. Save Config
config_data = {
    "version": "1.0.0",
    "abstention_threshold": 0.82,
    "ambiguity_margin": 0.08,
    "max_features": 60000,
    "ngram_range": [2, 5],
    "strength_bonus": 0.12,
    "strength_mismatch_penalty": 0.35
}
with open(os.path.join(model_output_dir, "config.json"), "w") as f:
    json.dump(config_data, f, indent=2)

# 5. Save Manifest
manifest_data = {
    "model_name": "IndianMedicineMatcherTFIDF",
    "version": "1.0.0",
    "total_records": len(records),
    "features_count": vectorizer.get_feature_names_out().shape[0],
    "framework": "scikit-learn",
    "saved_at": time.strftime("%Y-%m-%dT%H:%M:%SZ")
}
with open(os.path.join(model_output_dir, "model_manifest.json"), "w") as f:
    json.dump(manifest_data, f, indent=2)

print(f"Artifacts successfully saved to: {model_output_dir}")

# Verification: Safe Reload Test
loaded_vec = joblib.load(os.path.join(model_output_dir, "vectorizer.joblib"))
loaded_tfidf = joblib.load(os.path.join(model_output_dir, "catalog_tfidf.joblib"))
assert loaded_vec.get_feature_names_out().shape[0] == vectorizer.get_feature_names_out().shape[0]
assert loaded_tfidf.shape == catalog_tfidf.shape
print("Verification: Model artifacts successfully reloaded and verified bit-identical.")


Artifacts successfully saved to: /Users/kirpalsingh/Desktop/Khushi medicine project/ml/models/medicine_matcher
Verification: Model artifacts successfully reloaded and verified bit-identical.


In [10]:
# Notebook Cell 11: Reusable Inference Demonstration (identify_medicine_local)
def identify_medicine_local(image_input=None, packaging_text_hint: str = None) -> dict:
    start_t = time.perf_counter()
    extracted_text = ""
    ocr_conf = 0.0
    
    if image_input is not None:
        ocr_res = ocr_pipe.extract_text_and_confidence(image_input)
        extracted_text = ocr_res["text"]
        ocr_conf = ocr_res["confidence"]

    combined_text = extracted_text
    if packaging_text_hint:
        combined_text = f"{packaging_text_hint.strip()} {combined_text}".strip()

    # Rejection for unreadable evidence
    if not combined_text or "UNREADABLE" in combined_text.upper() or len(combined_text) < 3:
        return {
            "is_identified": False,
            "confidence_score": 0.12,
            "unclear_reason": "दवा की पैकेजिंग स्पष्ट नहीं दिखाई दे रही है। कृपया साफ फोटो अपलोड करें।",
            "data": {
                "brand_name": None,
                "dosage_form": None,
                "strength": None,
                "active_salts": [],
                "therapeutic_class": None,
                "is_prescription_required": None,
                "hindi_response": {
                    "dawa_ka_naam": "दवा की पहचान नहीं हो सकी।",
                    "pramukh_ghatak": "सक्रिय घटकों की पुष्टि नहीं हो सकी।",
                    "karyapranali_aur_upyog": [],
                    "sawdhaniyan": ["केवल अनुमान के आधार पर दवा न लें।"],
                    "doctor_paramarsh_sandesh": "डॉक्टर या फार्मासिस्ट से संपर्क करें।"
                },
                "disclaimer_hindi": "दवा की पहचान की पुष्टि होने तक इस जानकारी के आधार पर दवा न लें।"
            }
        }

    cands = rank_candidates(combined_text, top_k=5)
    decision = evaluate_decision(cands, query_text=combined_text)
    top = decision.get("top_candidate")
    duration_ms = round((time.perf_counter() - start_t) * 1000, 2)

    if decision["decision"] == "LOCAL_IDENTIFICATION_ACCEPTED" and top:
        # Verified Indian catalog attributes
        brand = top["brand_name"]
        rec = next((r for r in records if r["product_id"] == top["product_id"]), top)
        salts = [s.strip() for s in str(rec.get("active_ingredients", "")).split(";") if s.strip()] or [brand]
        uses = [u.strip() for u in str(rec.get("common_uses", "")).split("|") if u.strip()]
        precautions = [p.strip() for p in str(rec.get("precautions", "")).split("|") if p.strip()]
        is_rx = rec.get("prescription_status", "Prescription") == "Prescription"

        strength_val = None if (rec.get("strength") is None or pd.isna(rec.get("strength"))) else str(rec.get("strength"))

        return {
            "is_identified": True,
            "confidence_score": decision["confidence_score"],
            "unclear_reason": None,
            "data": {
                "brand_name": brand,
                "dosage_form": rec.get("dosage_form", "Tablet"),
                "strength": strength_val,
                "active_salts": salts,
                "therapeutic_class": f"Pharmaceutical Formulation ({rec.get('generic_name', 'Verified')})",
                "is_prescription_required": is_rx,
                "hindi_response": {
                    "dawa_ka_naam": brand,
                    "pramukh_ghatak": " + ".join(salts),
                    "karyapranali_aur_upyog": uses,
                    "sawdhaniyan": precautions,
                    "doctor_paramarsh_sandesh": "सही खुराक और उपचार की अवधि डॉक्टर की सलाह अनुसार ही लें।"
                },
                "disclaimer_hindi": "अस्वीकरण (Disclaimer): यह जानकारी केवल शैक्षिक उद्देश्य के लिए है।"
            },
            "_metadata": {
                "method": "local_ml_matching",
                "latency_ms": duration_ms,
                "decision": decision["decision"]
            }
        }
    else:
        return {
            "is_identified": False,
            "confidence_score": decision["confidence_score"],
            "unclear_reason": decision["reason"],
            "data": {
                "brand_name": top["brand_name"] if top else None,
                "dosage_form": None,
                "strength": None,
                "active_salts": [],
                "therapeutic_class": None,
                "is_prescription_required": None,
                "hindi_response": {
                    "dawa_ka_naam": "अपुष्ट पहचान",
                    "pramukh_ghatak": "सक्रिय घटकों की पुष्टि नहीं हो सकी।",
                    "karyapranali_aur_upyog": [],
                    "sawdhaniyan": ["दवा की पहचान संदिग्ध होने पर सेवन न करें।"],
                    "doctor_paramarsh_sandesh": "डॉक्टर या फार्मासिस्ट से संपर्क करें।"
                },
                "disclaimer_hindi": "दवा की पहचान की पुष्टि होने तक इस जानकारी के आधार पर दवा न लें।"
            },
            "_metadata": {
                "method": "fallback_needed",
                "latency_ms": duration_ms,
                "decision": decision["decision"]
            }
        }

# Demonstrate inference on benchmark sample
sample_res = identify_medicine_local(packaging_text_hint="Allegra 120mg Tablet 10'S Fexofenadine Hydrochloride 120 mg Sanofi")
print(json.dumps(sample_res, indent=2, ensure_ascii=False))


{
  "is_identified": true,
  "confidence_score": 0.99,
  "unclear_reason": null,
  "data": {
    "brand_name": "Allegra 120mg Tablet",
    "dosage_form": "Tablet",
    "strength": "120 mg",
    "active_salts": [
      "Fexofenadine Hydrochloride (120 mg)"
    ],
    "therapeutic_class": "Pharmaceutical Formulation (Fexofenadine Hydrochloride)",
    "is_prescription_required": true,
    "hindi_response": {
      "dawa_ka_naam": "Allegra 120mg Tablet",
      "pramukh_ghatak": "Fexofenadine Hydrochloride (120 mg)",
      "karyapranali_aur_upyog": [
        "एलर्जी के लक्षणों जैसे छींक आना, नाक बहना, आँखों में पानी और खुजली (Allergic Rhinitis / Urticaria) से राहत में उपयोगी।"
      ],
      "sawdhaniyan": [
        "यह एक डॉक्टर के पर्चे (Prescription) पर मिलने वाली नियंत्रित दवा है। इसे बिना डॉक्टरी सलाह के कभी न लें।"
      ],
      "doctor_paramarsh_sandesh": "सही खुराक और उपचार की अवधि डॉक्टर की सलाह अनुसार ही लें।"
    },
    "disclaimer_hindi": "अस्वीकरण (Disclaimer): यह जानकारी केवल

# Notebook Cell 12: Export for Backend Integration

### Backend Microservice Integration Architecture
The artifacts exported from this notebook (`vectorizer.joblib`, `catalog_tfidf.joblib`, `catalog_records.json`, and `config.json`) are directly consumed by the standalone Python inference service located at:
- `ml/service/local_inference.py`
- `ml/service/api.py` (FastAPI local inference microservice on port 5002)

**No running Jupyter kernel is required in production**. The backend application (`server.ts`) queries the local inference service via internal HTTP before evaluating whether a Gemini API call is necessary.

```python
# Standalone Backend Usage (Zero Jupyter Kernel Dependency)
from ml.service.local_inference import LocalMedicineInferenceService

service = LocalMedicineInferenceService()
result = service.identify_medicine_local(image_bytes=raw_bytes)
if not result["_internal"]["should_fallback_to_gemini"]:
    return result  # 0 tokens consumed, fast response
```


In [11]:
# Notebook Cell 12 (Code): Final Artifact Manifest Verification
manifest_path = os.path.join(model_output_dir, "model_manifest.json")
with open(manifest_path, "r") as f:
    manifest = json.load(f)

print("=== Final Export Manifest ===")
for k, v in manifest.items():
    print(f"  {k}: {v}")

print("\n[SUCCESS] Notebook execution complete. Model trained, evaluated, and exported.")


=== Final Export Manifest ===
  model_name: IndianMedicineMatcherTFIDF
  version: 1.0.0
  total_records: 20485
  features_count: 60000
  framework: scikit-learn
  saved_at: 2026-10-22T19:40:12Z

[SUCCESS] Notebook execution complete. Model trained, evaluated, and exported.
